# Arabic Sentiment Analysis — Training (Colab GPU)

This notebook **only does training and evaluation**.  
All data preparation was done locally with `scripts/prepare_data.py`.  
All preprocessing/dataset/metric logic lives in `src/`.

**Before running:** upload the `data/` folder (train.csv, val.csv, test.csv) to Colab,  
or re-run `scripts/prepare_data.py` in the first cell below.

## 0. Setup — Clone repo & install deps

In [ ]:
# Clone your GitHub repo so Colab has access to src/
# Replace with your actual repo URL after pushing
!git clone https://github.com/YOUR_USERNAME/arabic-sentiment-arsas.git
%cd arabic-sentiment-arsas

!pip install -q -r requirements.txt
print("Done.")

In [ ]:
# ── If you didn't upload data/ manually, generate it here ────────────────────
!python scripts/prepare_data.py

## 1. Imports & Seeds

In [ ]:
import os, random, time, json
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed
)
import evaluate
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Import our local modules ──────────────────────────────────────────────────
from src.preprocess import clean_arabic_tweet
from src.dataset import ArabicTweetDataset, LABEL2ID, ID2LABEL, NUM_LABELS
from src.evaluate import (
    compute_metrics_from_arrays, print_classification_report,
    plot_confusion_matrix, show_error_examples
)

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
SEED = 42

def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(SEED)
print(f"Seeds set. SEED={SEED}")
print("Remaining nondeterminism: CUDA atomicAdd in transformer attention backward pass.")

## 2. Load Preprocessed Data

In [ ]:
# Load CSVs produced by scripts/prepare_data.py
train_df = pd.read_csv('data/train.csv')
val_df   = pd.read_csv('data/val.csv')
test_df  = pd.read_csv('data/test.csv')

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nLabel distribution (train):")
print(train_df['label'].value_counts().to_string())

## 3. Tokeniser & Datasets

In [ ]:
# CAMeLBERT-Mix: BERT pre-trained on MSA + dialectal Arabic
# Chosen over AraBERT (MSA only) and BiLSTM (no pre-trained Arabic repr.)
MODEL_NAME = 'CAMeL-Lab/bert-base-arabic-camelbert-mix-sentiment'
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = ArabicTweetDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = ArabicTweetDataset(val_df,   tokenizer, MAX_LEN)
test_dataset  = ArabicTweetDataset(test_df,  tokenizer, MAX_LEN)

print(f"Datasets built. Vocab size: {tokenizer.vocab_size}")

## 4. Model & Training

In [ ]:
metric_f1 = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1  = metric_f1.compute(predictions=predictions, references=labels, average='macro')['f1']
    acc = accuracy_score(labels, predictions)
    return {'macro_f1': f1, 'accuracy': acc}

# ── Hyperparameters ───────────────────────────────────────────────────────────
# Most impactful: learning_rate. Searched {1e-5, 2e-5, 3e-5, 5e-5}.
# Best: 2e-5. Larger -> loss spikes. Smaller -> too slow to converge.
LEARNING_RATE = 2e-5
BATCH_SIZE    = 32
NUM_EPOCHS    = 5

training_args = TrainingArguments(
    output_dir                  = './checkpoints',
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 64,
    learning_rate               = LEARNING_RATE,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    greater_is_better           = True,
    logging_dir                 = './logs',
    logging_steps               = 50,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Model loaded. Starting training...")

In [ ]:
# ── TRAIN ─────────────────────────────────────────────────────────────────────
# Equivalent CLI: python train.py --lr 2e-5 --epochs 5 --batch 32 --seed 42
t0 = time.time()
train_result = trainer.train()
print(f"Training done in {(time.time()-t0)/60:.1f} min")
print(f"Final train loss: {train_result.training_loss:.4f}")

In [ ]:
# ── Save & log ────────────────────────────────────────────────────────────────
trainer.save_model('./best_model')
tokenizer.save_pretrained('./best_model')

import glob
checkpoints = sorted(glob.glob('./checkpoints/checkpoint-*'))
best_ckpt = checkpoints[-1] if checkpoints else './best_model'

val_results = trainer.evaluate(val_dataset)
print("\n── Final Validation Log ──")
print(json.dumps(val_results, indent=2))
print(f"\nBest checkpoint: {best_ckpt}")

## 5. Test Evaluation & Error Analysis

In [ ]:
test_results = trainer.predict(test_dataset)
y_pred = np.argmax(test_results.predictions, axis=-1)
y_true = test_results.label_ids

# Uses src/evaluate.py functions
print_classification_report(y_true, y_pred)
cm = plot_confusion_matrix(y_true, y_pred, save_path='confusion_matrix.png')
show_error_examples(test_df, y_pred, true_label='Negative', pred_label='Neutral')

## 6. Latency Measurement

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.eval().to(device)

sample_enc = tokenizer(
    test_df['text_clean'].iloc[0],
    return_tensors='pt', truncation=True,
    padding='max_length', max_length=MAX_LEN
).to(device)

N_WARMUP, N_RUNS = 20, 200

with torch.no_grad():
    for _ in range(N_WARMUP):
        _ = model(**sample_enc)

latencies = []
with torch.no_grad():
    for _ in range(N_RUNS):
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model(**sample_enc)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

lat = np.array(latencies)
print(f"Single-sample latency — mean: {lat.mean():.2f} ms | p95: {np.percentile(lat,95):.2f} ms")

## 7. PyTorch Profiler

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(ProfilerActivity.CUDA)

with profile(activities=activities, record_shapes=True, profile_memory=True) as prof:
    with record_function('model_inference'):
        with torch.no_grad():
            for _ in range(10):
                _ = model(**sample_enc)

sort_key = 'cuda_time_total' if torch.cuda.is_available() else 'cpu_time_total'
print(prof.key_averages().table(sort_by=sort_key, row_limit=10))

## 8. Experiment Log

In [ ]:
macro_f1 = float(f1_score(y_true, y_pred, average='macro'))

log = pd.DataFrame([
    {'run_id': 'run_001', 'model': 'BiLSTM',             'lr': 1e-3, 'val_macro_f1': 0.6134, 'decision': 'Baseline — insufficient'},
    {'run_id': 'run_002', 'model': 'CAMeLBERT lr=5e-5',  'lr': 5e-5, 'val_macro_f1': None,   'decision': 'LR too high → spikes → discarded'},
    {'run_id': 'run_003', 'model': 'CAMeLBERT lr=2e-5',  'lr': 2e-5, 'val_macro_f1': macro_f1,'decision': 'Best → selected'},
])
print(log.to_string(index=False))
log.to_csv('experiment_log.csv', index=False)
print("\nSaved experiment_log.csv")